# Twitter Sentiment Analysis with RNN/LSTM

Training on Sentiment140 dataset using GRU, LSTM, and hybrid architectures.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU'))
with tf.device('/GPU:0'):
    x = tf.random.normal((1000, 1000))
    print("GPU OK:", float(tf.reduce_sum(tf.matmul(x, x))))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU OK: -48896.90625


## Preprocessing

In [4]:
# COMPLETE TEST CELL v2 - drops empty sequences, run top to bottom
import os, re
import pandas as pd, numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

project_dir = '/content/drive/MyDrive/GitHub/Twitter-Sentiment-RNN'
os.chdir(project_dir)
dataset_path = 'data/training.1600000.processed.noemoticon.csv'
print(f"cwd: {os.getcwd()}  dataset exists: {os.path.exists(dataset_path)}")

df = pd.read_csv(dataset_path, encoding='latin-1',
                 names=['target','ids','date','flag','user','text'])
df['target'] = df['target'].replace(4, 1)

SAMPLE_SIZE = 160000
MAX_WORDS = 10000
MAX_LEN = 40
EPOCHS = 5
BATCH_SIZE = 256
df = df.sample(n=SAMPLE_SIZE, random_state=42)

def clean_tweet(t):
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'http\S+', '', t)
    t = re.sub(r'[^a-zA-Z\s]', '', t)
    return t.lower().strip()
df['text'] = df['text'].apply(clean_tweet)

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])

# drop empty sequences (tweets that cleaned down to nothing)
y_all = df['target'].values
keep = [i for i, s in enumerate(sequences) if len(s) > 0]
dropped = len(sequences) - len(keep)
sequences = [sequences[i] for i in keep]
y = y_all[keep]
print(f"dropped {dropped} empty sequences, {len(sequences):,} remain")

X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"train: {X_train.shape[0]:,}  test: {X_test.shape[0]:,}  vocab: {len(tokenizer.word_index):,}")

model = Sequential([
    Embedding(MAX_WORDS, 128, mask_zero=True),
    GRU(128),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=1)

cwd: /content/drive/MyDrive/GitHub/Twitter-Sentiment-RNN  dataset exists: True
dropped 394 empty sequences, 159,606 remain
train: 127,684  test: 31,922  vocab: 88,182
Epoch 1/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7584 - loss: 0.4989 - val_accuracy: 0.7892 - val_loss: 0.4517
Epoch 2/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8068 - loss: 0.4200 - val_accuracy: 0.7941 - val_loss: 0.4467
Epoch 3/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8236 - loss: 0.3877 - val_accuracy: 0.7896 - val_loss: 0.4564
Epoch 4/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8372 - loss: 0.3608 - val_accuracy: 0.7897 - val_loss: 0.4604
Epoch 5/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8513 - loss: 0.3339 - val_accuracy: 0.7828 - val_loss: 0.4836


In [ ]:
# Are labels correlated with text at all?
sample_positive = df[df['target'] == 1].sample(5)
sample_negative = df[df['target'] == 0].sample(5)

print("POSITIVE tweets:")
for text in sample_positive['text']:
    print(f"  {text}")

print("\nNEGATIVE tweets:")
for text in sample_negative['text']:
    print(f"  {text}")

POSITIVE tweets:
  @WeareTHATfamily hey we're headed there too 
  @kevjumba http://twitpic.com/3m93e - awww how cute!!  hope you're having fun with him out there and does he know you think he looks li ...
  Going to bed.  Nighty ~True Love Conquers All~ &lt;3
  We can finaly get weapons on TF2 with achievements 
  CHICOS diviertanse mucho, get fun so much, Peru is great and loves you!!!!! LOS QUEREMOS so MUCHHHHH!!! 

NEGATIVE tweets:
  exactly. see, i've turned for the worse since i dont have u in my life 24/7 now 
  Gah, I wish Twitter would block @asd123 already. It's ruining the #futuresummit stream (and the other trending streams) 
  well oh well, God must have something else for me 2 do this week, oh and my ankle hurts real bad,I twisted it! 
  just took a shower after i went swimming and now i am sooooo relaxed...but i still have a headache!!!!! 
  @charlottewhoax i feel like shite dno whyy, and i burnt my leg on the iron the other day  hahaha, clever me
